# Embeddings Explorer
Samples every table in the data pipeline, then deep-dives into each embedding run: paper counts, date ranges, and top papers by citation.

In [22]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import pandas as pd
from IPython.display import display, Markdown
from sqlalchemy import text

from literature_ai.core.db import ENGINE, load_table

## 1. Pipeline table snapshots

### `raw.raw_paper_searches`

In [23]:
df_papers = load_table('raw.raw_paper_searches')
print(f"Rows: {len(df_papers):,}")
df_papers[['paperId', 'title', 'year', 'venue', 'citationCount', 'isOpenAccess', 'collected_at']].head(5)

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

### `raw.full_papers`

In [ ]:
df_full = load_table('raw.full_papers')
print(f"Rows: {len(df_full):,}")
df_full.head(5)

Rows: 0


,paperId,full_text,pdf_url,collected_at


### `processed.processed_abstracts`

In [ ]:
df_proc = load_table('processed.processed_abstracts')
print(f"Rows: {len(df_proc):,}")
df_proc.head(5)

Rows: 0


,paperId,abstract_clean,abstract_length,word_count,has_formula,language,content_hash,processed_at


### `processed.embedding_runs_metadata`

In [ ]:
df_runs = load_table('processed.embedding_runs_metadata')
print(f"Rows: {len(df_runs):,}")
display(df_runs)

Rows: 1


,ran_at,embedding_model,embedding_version,n_dim,user_tags,source,run_id
0,2026-07-07 21:42:26.514381+00:00,specter_v2,None,768,{},collect,1


### `processed.abstract_embeddings`

In [ ]:
with ENGINE.connect() as conn:
    total = conn.execute(text('SELECT COUNT(*) FROM processed.abstract_embeddings')).scalar()
    df_emb_sample = pd.read_sql(
        'SELECT "paperId", run_id, processed_at, content_hash, embedding_768::text FROM processed.abstract_embeddings LIMIT 5',
        conn,
    )

print(f"Rows: {total:,}")
display(df_emb_sample)

Rows: 4,700


,paperId,run_id,processed_at,content_hash,embedding_768
0,2c03df8b48bf3fa39054345bafabfeff15bfd11d,1,2026-07-07 21:42:37.038022+00:00,b6e91e48dcd2d7b65c162f1f13c76f122711edc2177b94...,"[0.2029713,0.6437859,-0.3180334,0.32343253,-0...."
1,dc32a984b651256a8ec282be52310e6bd33d9815,1,2026-07-07 21:42:37.038022+00:00,8c876b291245a21818d93f169b084b03047fb5f285b0f9...,"[0.4027466,0.6113165,-0.1346894,-0.2396962,-0...."
2,5582bebed97947a41e3ddd9bd1f284b73f1648c2,1,2026-07-07 21:42:37.038022+00:00,f8af338ebfcbc66cc0d243c75d4a00e3a2c3e54de47a17...,"[0.014119079,0.79693395,-0.36396796,-0.3719218..."
3,4c75b748911ddcd888c5122f7672f69caa5d661f,1,2026-07-07 21:42:37.038022+00:00,9f067eb9eb59b7c48d8017f1b88da2ef44ed78158d6b5f...,"[-0.32550272,0.5053523,-0.24488288,-0.37329957..."
4,cab372bc3824780cce20d9dd1c22d4df39ed081a,1,2026-07-07 21:42:37.038022+00:00,065b65aee4717a95102391398502db3ff6153c64a24983...,"[0.3214283,0.43594941,-0.3490806,-0.2749592,-0..."


---
## 2. Embedding runs deep-dive

For each run in `embedding_runs_metadata`, show key metadata, paper count, processing window, and the top-cited papers that were embedded.

In [ ]:
df_runs_ordered = pd.read_sql(
    'SELECT * FROM processed.embedding_runs_metadata ORDER BY ran_at',
    ENGINE,
)

for _, run in df_runs_ordered.iterrows():
    run_id = str(run['run_id'])

    display(Markdown(f"### Run `{run_id}`"))
    display(Markdown(
        f"| Field | Value |\n"
        f"|---|---|\n"
        f"| Model | `{run['embedding_model']}` |\n"
        f"| Version | `{run['embedding_version'] or '—'}` |\n"
        f"| Dimensions | {run['n_dim']} |\n"
        f"| Source | `{run['source']}` |\n"
        f"| Tags | `{run['user_tags']}` |\n"
        f"| Ran at | {run['ran_at']} |"
    ))

    with ENGINE.connect() as conn:
        n_papers = conn.execute(
            text('SELECT COUNT(*) FROM processed.abstract_embeddings WHERE run_id = :rid'),
            {'rid': run_id},
        ).scalar()

        date_range = conn.execute(
            text('SELECT MIN(processed_at), MAX(processed_at) FROM processed.abstract_embeddings WHERE run_id = :rid'),
            {'rid': run_id},
        ).fetchone()

        df_top = pd.read_sql(
            text("""
                SELECT
                    r."paperId",
                    r.title,
                    r.year,
                    r.venue,
                    r."citationCount",
                    ae.processed_at
                FROM processed.abstract_embeddings ae
                JOIN raw.raw_paper_searches r ON ae."paperId" = r."paperId"
                WHERE ae.run_id = :rid
                ORDER BY r."citationCount" DESC NULLS LAST
                LIMIT 10
            """),
            conn,
            params={'rid': run_id},
        )

    print(f"Papers embedded : {n_papers:,}")
    print(f"Processed from  : {date_range[0]}")
    print(f"           to   : {date_range[1]}")
    print("\nTop 10 papers by citation count:")
    display(df_top)
    display(Markdown("---"))

### Run `e78398df-2c8a-4b6c-8bf7-ee8e89cbd43c`

| Field | Value |
|---|---|
| Model | `specter-v2` |
| Version | `—` |
| Dimensions | 768 |
| Source | `collect` |
| Tags | `{}` |
| Ran at | 2026-07-07 21:40:04.708959+00:00 |

Papers embedded : 0
Processed from  : None
           to   : None

Top 10 papers by citation count:


,paperId,title,year,venue,citationCount,processed_at


---

### Run `955fcaec-8a68-48ee-a5be-5ccb413ceb40`

| Field | Value |
|---|---|
| Model | `specter_v2` |
| Version | `—` |
| Dimensions | 768 |
| Source | `collect` |
| Tags | `{}` |
| Ran at | 2026-07-07 21:42:26.514381+00:00 |

Papers embedded : 4,700
Processed from  : 2026-07-07 21:42:37.038022+00:00
           to   : 2026-07-07 21:43:51.323181+00:00

Top 10 papers by citation count:


,paperId,title,year,venue,citationCount,processed_at
0,2c03df8b48bf3fa39054345bafabfeff15bfd11d,Deep Residual Learning for Image Recognition,2015,Computer Vision and Pattern Recognition,232590,2026-07-07 21:42:37.038022+00:00
1,dc32a984b651256a8ec282be52310e6bd33d9815,Highly accurate protein structure prediction w...,2021,Nature,37112,2026-07-07 21:42:37.038022+00:00
2,c8b25fab5608c3e033d34b4483ec47e68ba109b7,Swin Transformer: Hierarchical Vision Transfor...,2021,IEEE International Conference on Computer Vision,33446,2026-07-07 21:43:18.958425+00:00
3,23ffaa0fe06eae05817f527a47ac3291077f9e58,Rethinking the Inception Architecture for Comp...,2015,Computer Vision and Pattern Recognition,31298,2026-07-07 21:43:18.958425+00:00
4,5582bebed97947a41e3ddd9bd1f284b73f1648c2,Grad-CAM: Visual Explanations from Deep Networ...,2016,International Journal of Computer Vision,27803,2026-07-07 21:42:37.038022+00:00
5,4c75b748911ddcd888c5122f7672f69caa5d661f,Statistical Learning Theory,2021,Technometrics,22082,2026-07-07 21:42:37.038022+00:00
6,cab372bc3824780cce20d9dd1c22d4df39ed081a,DeepLab: Semantic Image Segmentation with Deep...,2016,IEEE Transactions on Pattern Analysis and Mach...,21170,2026-07-07 21:42:37.038022+00:00
7,846aedd869a00c09b40f1f1f35673cb22bc87490,Mastering the game of Go with deep neural netw...,2016,Nature,18992,2026-07-07 21:42:37.038022+00:00
8,c468bbde6a22d961829e1970e6ad5795e05418d1,The Unreasonable Effectiveness of Deep Feature...,2018,2018 IEEE/CVF Conference on Computer Vision an...,18413,2026-07-07 21:42:37.038022+00:00
9,d86084808994ac54ef4840ae65295f3c0ec4decd,Physics-informed neural networks: A deep learn...,2019,Journal of Computational Physics,18022,2026-07-07 21:42:37.038022+00:00


---